# How to Use Noise-Aware Preprocessing

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/noise_aware_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fnoise_aware_preprocessing.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/noise_aware_preprocessing.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


On NISQ hardware, qubits have different error rates and connectivity constraints. `NoiseAwarePreprocessor` assigns high-variance features to the most reliable qubits and remaps angles away from gate-error hotspots.

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning
from quprep.core.dataset import Dataset
from quprep.preprocess.noise_aware import NoiseAwarePreprocessor, NoiseProfile

print(f"quprep {qd.__version__}")

rng = np.random.default_rng(0)
X = np.column_stack([
    rng.normal(0.5, 2.0, 40),   # feature 0: high variance
    rng.normal(0.5, 0.1, 40),   # feature 1: low variance
    rng.normal(0.5, 1.5, 40),   # feature 2: medium variance
    rng.normal(0.5, 0.05, 40),  # feature 3: very low variance
])
ds = Dataset(data=X)
print(f"Feature variances : {np.var(X, axis=0).round(3)}")

## 1. Define a hardware noise profile

In [ ]:
noise = NoiseProfile(
    qubit_error_rates=[0.001, 0.003, 0.002, 0.008],  # qubit 3 is noisiest
    coupling_map=[(0, 1), (1, 2), (2, 3)],            # linear chain
)
print(f"Qubit error rates : {noise.qubit_error_rates}")
print(f"Connectivity      : {noise.coupling_map}")

## 2. Fit the preprocessor — qubit assignment

In [ ]:
nap = NoiseAwarePreprocessor(noise_profile=noise)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    ds_out = nap.fit_transform(ds)

print(f"Feature variances : {np.var(X, axis=0).round(3)}")
print(f"Qubit error rates : {noise.qubit_error_rates}")
print(f"Qubit assignment  : {nap.qubit_assignment_}")
print("(high-variance features → low-error qubits)")

## 3. SWAP overhead

In [ ]:
print(f"SWAP gates before : {nap.estimated_swaps_before_}")
print(f"SWAP gates after  : {nap.estimated_swaps_after_}")

## 4. Angle dead-zone

In [ ]:
print(f"Dead-zone width : {nap.angle_deadzone}")
print(f"Output range    : [{ds_out.data.min():.3f}, {ds_out.data.max():.3f}]")

## 5. Encode the noise-aware dataset

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = qd.Pipeline(
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
    ).fit_transform(ds_out)

print(f"Circuits : {len(result.encoded)}")
print(f"Qubits   : {result.encoded[0].metadata['n_qubits']}")
print(qd.draw_ascii(result.encoded[0]))